# Loading

In [ ]:
import omicverse as ov
import pandas as pd
import json
import numpy as np
from adjustText import adjust_text
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
ov.plot_set(font_path='Arial')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42


In [ ]:
import os

# To do list 1

In [ ]:
df = pd.read_csv("Process_Data/to_do_list_1/brain_regionH2_celltyep_fraction.csv",index_col=[0])
df = df.loc[:,['ChP','GABA neuron','GLU neuron','Microglia','NPC','OPC','VLMC','IPC']]
if 'Hippocampus' in df.index:
    df = df.rename(index={'Hippocampus': 'Hipp'})
df

In [ ]:
if 'IPC' in df.columns:
    df = df.drop(columns=['IPC'])
df = df.div(df.sum(axis=1), axis=0) * 100
df

In [ ]:
color_map = {
    'ChP':         '#EE934E',  # 橙色
    #'IPC':         '#AECDE1',  # 浅蓝
    'GABA neuron': '#D1352B',  # 红色
    'GLU neuron':  '#7DBFA7',  # 青绿色 (Teal)
    'Microglia':   '#D2EBC8',  # 浅绿
    'NPC':         '#3C77AF',  # 深蓝
    'OPC':         '#9B5B33',  # 棕色
    'VLMC':        '#F5CFE4'   # 浅粉色
}
fig, ax = plt.subplots(figsize=(4.5, 6), dpi=300)

df.plot(kind='barh', stacked=True, 
             color=[color_map[c] for c in df.columns], 
             width=0.9, ax=ax, edgecolor='none')

ax.set_xlim(0, 100)
ax.set_xlabel('Cell percentage (%)', fontsize=12, color='black')

ax.set_ylabel('') 

ax.tick_params(axis='y', labelsize=12, length=4, color='black')
ax.tick_params(axis='x', labelsize=12, length=4, color='black')

ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['top'].set_linewidth(1.5)
ax.spines['right'].set_linewidth(1.5)
ax.xaxis.set_tick_params(width=2)
ax.yaxis.set_tick_params(width=2)

handles, labels = ax.get_legend_handles_labels()
legend_dict = dict(zip(labels, handles))

display_order_for_mpl = [
    'GABA neuron', 'GLU neuron', 
    'NPC', 'OPC', 
    'VLMC', 'Microglia', 
    'ChP'
]

ordered_handles = [legend_dict[l] for l in display_order_for_mpl]
ordered_labels = display_order_for_mpl

ax.legend(ordered_handles, ordered_labels, 
          loc='upper center', 
          bbox_to_anchor=(0.4, -0.12), # 向下移动图例
          ncol=4, # 4列
          frameon=False, # 无边框
          fontsize=14,
          handlelength=0.8, # 色块大小
          handletextpad=0.2, # 色块和文字的间距
          columnspacing=0.5) # 列间距

plt.tight_layout()
plt.savefig('Figure/To_do_list_1/brain_region_celltype_fraction.pdf',dpi=300, bbox_inches='tight')
plt.show()


# To do list 2

## dpt pseodutime

In [ ]:
adata = sc.read_h5ad("Process_Data/to_do_list_2/obj_sc_20241104.h5ad")
adata

In [ ]:
cluster2annotation = {
    'Gaba': 'GABA neuron',
    'Glu': 'GLU neuron',
    'Immune cell': 'Microglia',
    'NPC': 'NPC',
    'Neuron': 'Unknown',
    'Non-neuron': 'ChP',
    'OPC': 'OPC',
    'VLMC': 'VLMC',
}

adata.obs['annotation_major'] = adata.obs['annotation_major'].map(cluster2annotation).astype('category')
adata.uns['annotation_major_colors'] = ['#D1352B', '#7DBFA7', '#D2EBC8', '#3C77AF','#dddddd','#EE934E','#9B5B33','#F5CFE4']

In [ ]:
fig, ax = plt.subplots(figsize=(5,5), dpi=300)
ov.utils.embedding(adata,basis='X_umap',
                   color=['annotation_major'],ax=ax,show=False,
                   frameon='small',cmap='Reds',wspace=0.55)
save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_2/celltype_umap.pdf'
plt.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)

In [ ]:
adata = adata[adata.obs['annotation_major'].isin(['GABA neuron','GLU neuron','NPC','IPC','OPC',])]

In [ ]:
sc.pp.neighbors(adata, use_rep='X_harmony', n_pcs=50)
sc.tl.diffmap(adata)
adata.uns["iroot"] = np.flatnonzero(adata.obs["annotation_major"] == "NPC")[0]
sc.tl.dpt(adata)

In [ ]:
sc.pl.umap(adata,color=['annotation_major','dpt_pseudotime'],wspace=0.5)

In [ ]:
fig, ax = plt.subplots(figsize=(5,5), dpi=300)
ov.utils.embedding(adata,basis='X_umap',
                   color=['dpt_pseudotime'],ax=ax,show=False,
                   frameon='small',cmap='Reds',wspace=0.55)
save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_2/dpt_pseudotime.pdf'
plt.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)

In [ ]:
# from matplotlib.pyplot import rc_context

with rc_context({"figure.figsize": (5, 5)}):
    ax = sc.pl.violin(adata, ["dpt_pseudotime"], groupby="annotation_major",size=0, show=False)
    plt.xticks(rotation=45, ha='right')
    plt.xlabel("")
    save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_2/dpt_pseudotime_violin.pdf'
    plt.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)

# Gene expression

In [ ]:
adata = adata[adata.obs['annotation_major']!='Unknown']
adata

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
genes_to_plot = [
    'VIM', 'FABP7', 'SOX2', 'TOP2A',             # NPC
    'NEUROD6', 'NEUROD2', 'SLC17A6', 'SLC17A7',  # GLU neuron
    'ERBB4', 'DLX5', 'GAD1', 'GAD2',             # GABA neuron
    'SPP1', 'CCL4', 'CCL3', 'CCL3L1',            # Microglia
    'FN1', 'COL1A2', 'LAMA2', 'DLC1',            # VLMC
    'TTR', 'HBG2', 'HBG1', 'IGFBP7',             # ChP
    'OLIG1', 'OLIG2', 'SOX8', 'PCDH15'           # OPC
]

cell_type_order = [
    'NPC', 'GLU neuron', 'GABA neuron', 'Microglia', 
    'VLMC', 'ChP',  'OPC'
]

# 定义注释列的名称
annotation_col = 'annotation_major'

# =============================
# 2. 数据检查与预处理 (重要)
# =============================

# [检查 1] 确保注释列存在
if annotation_col not in adata.obs.columns:
    raise ValueError(f"在 adata.obs 中找不到列名 '{annotation_col}'，请检查你的数据。")

# [检查 2] 筛选存在的基因
# 有些基因可能在你的数据集中被过滤掉了，这里只保留存在的基因，防止报错
genes_present = [g for g in genes_to_plot if g in adata.var_names]
genes_missing = set(genes_to_plot) - set(genes_present)
if genes_missing:
    print(f"提示: 以下基因在 adata 中未找到，将被跳过: {genes_missing}")

# [检查 3] 设置细胞类型的顺序
# 将注释列转换为分类类型，并指定顺序。这确保了 X 轴的排列与图片一致。
# 首先确保只使用数据中实际存在的细胞类型
existing_cell_types = [ct for ct in cell_type_order if ct in adata.obs[annotation_col].unique()]

adata.obs[annotation_col] = adata.obs[annotation_col].astype('category')
# 重新排序类别
adata.obs[annotation_col] = adata.obs[annotation_col].cat.reorder_categories(existing_cell_types)


# =============================
# 3. 绘图
# =============================

# 创建 Dot plot
# swap_axes=True 是关键，因为 scanpy 默认基因在 X 轴，细胞在 Y 轴，这与目标图片相反。
dp = sc.pl.dotplot(
    adata,
    var_names=genes_present,        # 使用筛选后存在的基因列表
    groupby=annotation_col,         # 按照你的细胞类型注释列分组
    standard_scale='var',           # 重要：按基因标准化表达量(0-1之间)，使不同表达水平的基因具有可比性
    cmap='OrRd',                    # 使用橙红色渐变色系 (Orange-Red)，与原图接近
    colorbar_title='Average Expression', # 设置颜色条标题
    size_title='Percent Expressed',      # 设置点大小图例标题
    swap_axes=True,                 # 交换坐标轴：让基因在Y轴，细胞类型在X轴
    figsize=(5,8),
    show=False,                     # 不立即显示，以便进行后续调整或保存
    vmax=2,
    return_fig=True                 # 返回图像对象
)

# 获取 axes 字典，方便我们操作各个部分
axes_dict = dp.get_axes()
ax_main = axes_dict['mainplot_ax']
ax_size_legend = axes_dict['size_legend_ax']
ax_color_legend = axes_dict['color_legend_ax']

# 修正主图样式 (保留你之前的修改)
ax_main.set_xticklabels(ax_main.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
plt.setp(ax_main.get_yticklabels(), fontstyle='italic')
ax_main.spines['top'].set_visible(False)
ax_main.spines['right'].set_visible(False)
ax_main.spines['bottom'].set_linewidth(2)
ax_main.spines['left'].set_linewidth(2)
ax_main.tick_params(axis='both', width=2, length=6)

ax_color_legend = axes_dict['color_legend_ax']
new_pos = [0.8, 0.2, 0.15, 0.03] 
ax_color_legend.set_position(new_pos)

# =============================
# 4. 显示最终结果
# =============================
plt.tight_layout() # 自动调整间距
save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_2/dotplot_marker.pdf'
plt.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)
plt.show()

In [ ]:
help(sc.pl.dotplot)

## CytoTRACE2

In [ ]:
adata = sc.read_h5ad("Process_Data/to_do_list_2/obj_sc_20241104.h5ad")
adata

In [ ]:
cluster2annotation = {
    'Gaba': 'GABA neuron',
    'Glu': 'GLU neuron',
    'Immune cell': 'Microglia',
    'NPC': 'NPC',
    'Neuron': 'IPC',
    'Non-neuron': 'ChP',
    'OPC': 'OPC',
    'VLMC': 'VLMC',
}

adata.obs['annotation_major'] = adata.obs['annotation_major'].map(cluster2annotation).astype('category')
adata.uns['annotation_major_colors'] = ['#D1352B', '#7DBFA7', '#D2EBC8', '#3C77AF','#AECDE1','#EE934E','#9B5B33','#F5CFE4']

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat_v3",
)
adata

In [ ]:
results =  ov.single.cytotrace2(adata,
    use_model_dir="Process_Data/to_do_list_2/5_models_weights",
    species="human",
    batch_size = 5000,
    smooth_batch_size = 1000,
    disable_parallelization = False,
    max_cores = 2,
    max_pcs = 200,
    seed = 14,
    output_dir = 'Process_Data/to_do_list_2/cytotrace2_results'
)

In [ ]:
ov.utils.embedding(adata,basis='X_umap',
                   color=['annotation_major','CytoTRACE2_Score'],
                   frameon='small',cmap='Reds',wspace=0.55)

In [ ]:
fig, ax = plt.subplots(figsize=(5,5), dpi=300)
ov.utils.embedding(adata,basis='X_umap',
                   color=['CytoTRACE2_Score'],ax=ax,show=False,
                   frameon='small',cmap='Reds',wspace=0.55)
save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_2/CytoTRACE2_score.pdf'
plt.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)

In [ ]:
CytoTRACE2_Score = pd.read_csv("Process_Data/to_do_list_2/cytotrace2_results/cytotrace2_results.txt",sep="\t",index_col=[0])
CytoTRACE2_Score

In [ ]:
CytoTRACE2_Score.columns

In [ ]:
adata.obs.loc[:,['preKNN_CytoTRACE2_Score', 'preKNN_CytoTRACE2_Potency',
       'CytoTRACE2_Score', 'CytoTRACE2_Potency', 'CytoTRACE2_Relative']] = CytoTRACE2_Score.loc[adata.obs_names,:]

In [ ]:
adata.obs

In [ ]:
ov.utils.embedding(adata,basis='X_umap',
                   color=['annotation_major','CytoTRACE2_Potency','CytoTRACE2_Score','CytoTRACE2_Relative'],ncols=2,
                   frameon='small',cmap='Reds',wspace=0.55)

In [ ]:
# from matplotlib.pyplot import rc_context

with rc_context({"figure.figsize": (5, 5)}):
    ax = sc.pl.violin(adata, ["CytoTRACE2_Score"], groupby="annotation_major",size=0, show=False)
    plt.xticks(rotation=45, ha='right')
    plt.xlabel("")
    save_path = '/home/dataset-assist-0/yaosen/lihan/hulei/Analysis/fetal_brain/Figure/To_do_list_2/CytoTRACE2_Score_violin.pdf'
    plt.savefig(save_path, bbox_inches='tight', format='pdf', dpi=300)